# 03 - Live Room Localization GUI (SSID-only)

Detection live de salle basee sur les SSID environnants.

In [ ]:
# Optionnel
# %pip install -q pywifi pandas numpy scikit-learn joblib

In [ ]:
from pathlib import Path
import threading
import time
from collections import deque
import tkinter as tk
from tkinter import ttk, messagebox

import numpy as np
import pywifi

from robust_localization import load_artifacts, normalize_ssid, predict_scan

In [ ]:
ARTIFACT_DIR = Path('./artifacts_robust_ssid')
assets = load_artifacts(ARTIFACT_DIR)
model = assets['model']
feature_builder = assets['feature_builder']
label_encoder = assets['label_encoder']

print('Artefacts charges:', ARTIFACT_DIR.resolve())

In [ ]:
class WiFiScanner:
    def __init__(self):
        wifi = pywifi.PyWiFi()
        interfaces = wifi.interfaces()
        if not interfaces:
            raise RuntimeError('Aucune interface Wi-Fi detectee.')
        self.iface = interfaces[0]

    def scan_once(self, wait_seconds=0.8):
        self.iface.scan()
        time.sleep(wait_seconds)
        results = self.iface.scan_results()

        # SSID-only: on garde le meilleur RSSI par SSID
        scan = {}
        for net in results:
            ssid = normalize_ssid(net.ssid)
            if ssid == '<hidden>':
                continue
            signal = float(net.signal)
            if ssid not in scan or signal > scan[ssid]:
                scan[ssid] = signal
        return scan

In [ ]:
class LiveLocalizerApp:
    def __init__(self, root, model, feature_builder, label_encoder):
        self.root = root
        self.model = model
        self.feature_builder = feature_builder
        self.label_encoder = label_encoder

        self.scanner = WiFiScanner()
        self.running = False
        self.scan_interval = 1.5
        self.scan_history = deque(maxlen=5)

        self._build_ui()

    def _build_ui(self):
        self.root.title('Live Room Localizer - SSID')
        self.root.geometry('700x420')

        title = ttk.Label(self.root, text='Localisation Live de Salle (SSID)', font=('Segoe UI', 16, 'bold'))
        title.pack(pady=10)

        controls = ttk.Frame(self.root)
        controls.pack(pady=5)

        self.start_btn = ttk.Button(controls, text='Start', command=self.start)
        self.start_btn.grid(row=0, column=0, padx=8)
        self.stop_btn = ttk.Button(controls, text='Stop', command=self.stop, state='disabled')
        self.stop_btn.grid(row=0, column=1, padx=8)

        ttk.Label(controls, text='Interval (s):').grid(row=0, column=2, padx=8)
        self.interval_var = tk.StringVar(value='1.5')
        self.interval_entry = ttk.Entry(controls, textvariable=self.interval_var, width=8)
        self.interval_entry.grid(row=0, column=3, padx=8)

        self.pred_label = ttk.Label(self.root, text='Salle predite: -', font=('Segoe UI', 14, 'bold'))
        self.pred_label.pack(pady=10)
        self.conf_label = ttk.Label(self.root, text='Confiance: -', font=('Segoe UI', 12))
        self.conf_label.pack(pady=4)
        self.ap_label = ttk.Label(self.root, text='SSID visibles: -', font=('Segoe UI', 12))
        self.ap_label.pack(pady=4)

        self.top_text = tk.Text(self.root, height=10, width=80)
        self.top_text.pack(pady=10)

    def start(self):
        try:
            self.scan_interval = max(0.5, float(self.interval_var.get()))
        except ValueError:
            messagebox.showerror('Erreur', 'Interval invalide.')
            return

        self.running = True
        self.start_btn.config(state='disabled')
        self.stop_btn.config(state='normal')

        t = threading.Thread(target=self._loop, daemon=True)
        t.start()

    def stop(self):
        self.running = False
        self.start_btn.config(state='normal')
        self.stop_btn.config(state='disabled')

    def _merge_history(self):
        if not self.scan_history:
            return {}

        keys = set()
        for d in self.scan_history:
            keys.update(d.keys())

        merged = {}
        for k in keys:
            vals = [d[k] for d in self.scan_history if k in d]
            merged[k] = float(np.mean(vals))
        return merged

    def _loop(self):
        while self.running:
            try:
                scan = self.scanner.scan_once(wait_seconds=0.8)
                self.scan_history.append(scan)
                merged_scan = self._merge_history()

                if not merged_scan:
                    time.sleep(self.scan_interval)
                    continue

                result = predict_scan(
                    scan_dict=merged_scan,
                    model=self.model,
                    feature_builder=self.feature_builder,
                    label_encoder=self.label_encoder,
                    top_k=5,
                )

                self.root.after(0, self._refresh_ui, result, len(scan))
            except Exception as e:
                self.root.after(0, lambda: messagebox.showerror('Erreur scan', str(e)))
                self.running = False

            time.sleep(self.scan_interval)

    def _refresh_ui(self, result, ssid_count):
        self.pred_label.config(text=f"Salle predite: {result['predicted_room']}")
        self.conf_label.config(text=f"Confiance: {result['confidence']:.3f}")
        self.ap_label.config(text=f"SSID visibles (scan courant): {ssid_count}")

        lines = ['Top predictions:']
        for item in result['top_predictions']:
            lines.append(f"- {item['room']}: {item['probability']:.3f}")

        self.top_text.delete('1.0', tk.END)
        self.top_text.insert(tk.END, '\n'.join(lines))

In [ ]:
root = tk.Tk()
app = LiveLocalizerApp(root, model, feature_builder, label_encoder)
root.mainloop()